<a href="https://colab.research.google.com/github/aligreo/TriEncoder-Unet-Project/blob/main/msseg_preprocessing_archeticture_final_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## installing necessery packages

In [1]:
!uv pip install SimpleITK monai gdown

Using Python 3.12.13 environment at: /usr
Resolved 41 packages in 445ms                                        
Prepared 1 package in 142ms                                              
Installed 1 package in 18ms                                 
 + monai==1.6.0


## package imports

In [3]:
import warnings
import os
import glob
import zipfile
import random
from pathlib import Path
import torch
import numpy as np

# Suppress specific warnings to clean up output
warnings.filterwarnings("ignore", category=UserWarning, message=".*non-tuple sequence for multidimensional indexing.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*cuda.cudart module is deprecated.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*monai.transforms.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*always_return_as_numpy.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*ground truth of class 0 is all 0.*")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Load MSSEG dataset

In [4]:
MSSEG_TRAIN_ZIP = "MSSEG-Training.zip"
MSSEG_TEST_ZIP = "MSSEG-Testing.zip"
MSSEG_EXTRACT_DIR = "MSSEG-Training"
MSSEG_EXTRACT_DIR_TEST = "MSSEG-Testing"

def unzip_if_needed(zip_path, extract_dir):
    if zip_path and os.path.exists(zip_path):
        marker = Path(extract_dir) / ".extracted"
        if not marker.exists():
            os.makedirs(extract_dir, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)
            marker.touch()
        return extract_dir
    return extract_dir

def first_match(files, include_terms, exclude_terms=(), path_must_include=None):
    include_terms = [t.upper() for t in include_terms]
    exclude_terms = [t.upper() for t in exclude_terms]
    candidates = []
    for f in files:
        name = os.path.basename(f).upper()
        full_path_upper = f.upper()
        if path_must_include and path_must_include.upper() not in full_path_upper:
            continue
        if all(t in name for t in include_terms) and not any(t in name for t in exclude_terms):
            candidates.append(f)
    return sorted(candidates)[0] if candidates else None

def find_case_files(nii_files, case_log_id="Unknown Case"):
    if not nii_files:
        print(f"  [Warning] No NIfTI files found at all for {case_log_id}")
        return None

    flair = first_match(nii_files, ["FLAIR"], path_must_include="PREPROCESSED_DATA")
    t2 = first_match(nii_files, ["T2"], exclude_terms=["T2STAR", "T2_STAR"], path_must_include="PREPROCESSED_DATA")

    t1_exclude = ["GADO", "GD", "GAD", "CONTRAST", "FLAIR", "T2", "CONSENSUS", "MASK", "SEG", "GT"]
    t1 = first_match(nii_files, ["T1"], t1_exclude, path_must_include="PREPROCESSED_DATA")

    if not t1:
        dp_fallback = first_match(nii_files, ["DP"], t1_exclude, path_must_include="PREPROCESSED_DATA")
        if dp_fallback:
            print(f"  [CRITICAL WARNING] T1 is missing for {case_log_id}, and DP scan was found. Skipping case to avoid Channel Corruption!")

    label = (first_match(nii_files, ["CONSENSUS"]) or first_match(nii_files, ["LESION"]) or first_match(nii_files, ["GT"]))

    if not (flair and t1 and t2 and label):
        missing = []
        if not flair: missing.append("FLAIR")
        if not t1: missing.append("T1")
        if not t2: missing.append("T2")
        if not label: missing.append("LABEL/CONSENSUS")
        print(f"  [Dropped Case] {case_log_id} is missing modalities: {missing}")
        return None

    return {"flair": flair, "t1": t1, "t2": t2, "label": label}

def collect_dataset(root_dir, source_name):
    root_dir = str(root_dir)
    if not os.path.exists(root_dir): return []
    patient_map = {}

    for root, dirs, files in os.walk(root_dir):
        nii_in_dir = [os.path.join(root, f) for f in files if f.lower().endswith(('.nii', '.nii.gz'))]
        if not nii_in_dir: continue

        parts = Path(root).parts
        patient_folder = next((p for p in parts if "PATIENT_" in p.upper()), None)

        if patient_folder:
            idx = parts.index(patient_folder)
            center_folder = parts[idx-1] if idx > 0 else "Center_Unknown"
            key = f"{center_folder}_{patient_folder}"
            if key not in patient_map: patient_map[key] = []
            patient_map[key].extend(nii_in_dir)

    final_cases = []
    for key, all_files in patient_map.items():
        case_log_id = f"{source_name}_{key}"
        item = find_case_files(all_files, case_log_id=case_log_id)
        if item:
            item["source"] = source_name
            item["case_id"] = case_log_id
            final_cases.append(item)

    return sorted(final_cases, key=lambda x: x['case_id'])

msseg_root = unzip_if_needed(MSSEG_TRAIN_ZIP, MSSEG_EXTRACT_DIR)
msseg_test_root = unzip_if_needed(MSSEG_TEST_ZIP, MSSEG_EXTRACT_DIR_TEST)

msseg_train_files = collect_dataset(msseg_root, "MSSEG_TRAIN")
msseg_test_files = collect_dataset(msseg_test_root, "MSSEG_TEST")

print(f"MSSEG train cases found: {len(msseg_train_files)}")
print(f"MSSEG test cases found: {len(msseg_test_files)}")

MSSEG train cases found: 15
MSSEG test cases found: 38


## Data Preprocessing & SplitSplit

In [5]:
from sklearn.model_selection import train_test_split
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    NormalizeIntensityd, ConcatItemsd, DeleteItemsd,
    RandCropByPosNegLabeld, EnsureTyped, CropForegroundd,
    RandFlipd, RandRotate90d, RandScaleIntensityd, RandShiftIntensityd,
    RandGaussianNoised, RandBiasFieldd, RandAdjustContrastd, Lambdad,
)
from monai.data import PersistentDataset, DataLoader
import os

def binarize_label(x):
    return (x > 0).astype(x.dtype)

base_transforms = [
    LoadImaged(keys=["flair", "t1", "t2", "label"]),
    EnsureChannelFirstd(keys=["flair", "t1", "t2", "label"]),
    Orientationd(keys=["flair", "t1", "t2", "label"], axcodes="RAS"),
    Spacingd(
        keys=["flair", "t1", "t2", "label"],
        pixdim=(1.0, 1.0, 1.0),
        mode=("bilinear", "bilinear", "bilinear", "nearest"),
        padding_mode="zeros",
    ),
    Lambdad(keys="label", func=binarize_label),
    CropForegroundd(keys=["flair", "t1", "t2", "label"], source_key="flair"),
    NormalizeIntensityd(keys=["flair", "t1", "t2"], nonzero=True, channel_wise=True),
    ConcatItemsd(keys=["flair", "t1", "t2"], name="image", dim=0),
    DeleteItemsd(keys=["flair", "t1", "t2"]),
]

num_samples_per_image = 6
train_transforms = Compose(base_transforms + [
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label",
        spatial_size=(96, 96, 96), pos=5, neg=1,
        num_samples=num_samples_per_image, image_key="image", image_threshold=0,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.25, max_k=3),
    RandScaleIntensityd(keys="image", factors=0.2, prob=0.5),
    RandShiftIntensityd(keys="image", offsets=0.15, prob=0.5),
    RandGaussianNoised(keys="image", prob=0.15, mean=0.0, std=0.01),
    RandBiasFieldd(keys="image", prob=0.15, coeff_range=(0.0, 0.03)),
    RandAdjustContrastd(keys="image", prob=0.2, gamma=(0.8, 1.2)),
    EnsureTyped(keys=["image", "label"]),
])

val_transforms = Compose(base_transforms + [EnsureTyped(keys=["image", "label"])])
test_transforms = Compose(base_transforms + [EnsureTyped(keys=["image", "label"])])

# Randomly choose 12 Train and 3 Val from the 15 train files
shuffled_files = random.sample(msseg_train_files, len(msseg_train_files))
train_files = shuffled_files[:12]
val_files = shuffled_files[12:15]
test_files = msseg_test_files

print(f"--- Dataset Statistics ---")
print(f"Initial volumes for Train: {len(train_files)}")
print(f"Initial volumes for Val: {len(val_files)}")
print(f"Initial volumes for Test: {len(test_files)}")

cache_dir_train = "/content/persistent_cache_train"
cache_dir_val = "/content/persistent_cache_val"
cache_dir_test = "/content/persistent_cache_test"

os.makedirs(cache_dir_train, exist_ok=True)
os.makedirs(cache_dir_val, exist_ok=True)
os.makedirs(cache_dir_test, exist_ok=True)

train_ds = PersistentDataset(data=train_files, transform=train_transforms, cache_dir=cache_dir_train)
val_ds = PersistentDataset(data=val_files, transform=val_transforms, cache_dir=cache_dir_val)
test_ds = PersistentDataset(data=test_files, transform=test_transforms, cache_dir=cache_dir_test)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

--- Dataset Statistics ---
Initial volumes for Train: 12
Initial volumes for Val: 3
Initial volumes for Test: 38


## baselines

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets import UNet, AttentionUnet, BasicUNetPlusPlus

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Baseline 1: Standard 3D U-Net (Early Fusion - 3 Channels Concatenated)
def get_standard_unet3d():
    return UNet(
        spatial_dims=3,
        in_channels=3,
        out_channels=1,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm="INSTANCE",
        dropout=0.10,
    ).to(device)


# Baseline 2: 3D Attention U-Net (Oktay et al., 2018)
def get_attention_unet3d():
    return AttentionUnet(
        spatial_dims=3,
        in_channels=3,
        out_channels=1,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        dropout=0.10,
    ).to(device)


# Baseline 3: 3D U-Net++ (Nested Dense Skip Pathways)
def get_unet_plusplus3d():
    return BasicUNetPlusPlus(
        spatial_dims=3,
        in_channels=3,
        out_channels=1,
        features=(16, 32, 64, 128, 256, 16),
        dropout=0.10,
    ).to(device)

## Training script

In [7]:
import os
import math
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric, ConfusionMatrixMetric
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete

standard_criterion = DiceFocalLoss(sigmoid=True, squared_pred=True, lambda_dice=0.5, lambda_focal=0.5)

def train_and_eval_baseline(model, model_name, train_loader, val_loader, test_loader, epochs=150):
    print(f"\n{'='*30}\n🚀 Starting Training for Baseline: {model_name}\n{'='*30}")

    save_weight_path = f"best_{model_name}.pth"
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)

    accumulation_steps = 2
    steps_per_epoch = math.ceil(len(train_loader) / accumulation_steps)
    total_steps = steps_per_epoch * epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=2e-4, total_steps=total_steps, pct_start=0.3)
    scaler = torch.amp.GradScaler("cuda")

    dice_metric = DiceMetric(include_background=False, reduction="mean")
    post_pred = AsDiscrete(threshold=0.5)

    best_val_dice = 0
    scheduler_step_count = 0

    def model_predictor(x):
        out = model(x)
        return out[-1] if isinstance(out, (list, tuple)) else out

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader):
            inputs = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(inputs)

                if isinstance(logits, (list, tuple)):
                    loss = sum(standard_criterion(l, labels) for l in logits) / (len(logits) * accumulation_steps)
                else:
                    loss = standard_criterion(logits, labels) / accumulation_steps

            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=12.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scheduler_step_count < total_steps:
                    scheduler.step()
                    scheduler_step_count += 1

        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            model.eval()
            dice_metric.reset()
            with torch.no_grad():
                for val_batch in val_loader:
                    v_in = val_batch["image"].to(device, non_blocking=True)
                    v_lbl = val_batch["label"].to(device, non_blocking=True)
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                        v_logits = sliding_window_inference(
                            v_in, roi_size=(96, 96, 96), sw_batch_size=2,
                            predictor=model_predictor, overlap=0.25, mode="gaussian"
                        )
                    v_pred = [post_pred(p) for p in torch.sigmoid(v_logits)]
                    dice_metric(y_pred=v_pred, y=v_lbl)

            current_dice = dice_metric.aggregate().item()
            dice_metric.reset()

            if current_dice > best_val_dice:
                best_val_dice = current_dice
                torch.save(model.state_dict(), save_weight_path)
                print(f"  [Epoch {epoch+1}] New Best Checkpoint Saved! Val Dice: {best_val_dice:.4f}")

            if (epoch + 1) % 20 == 0:
                print(f"Epoch [{epoch+1:03d}/{epochs:03d}] - Current Val Dice: {current_dice:.4f} (Best: {best_val_dice:.4f})")

    # Testing
    print(f"\nEvaluating {model_name} on Test Cases...")
    if os.path.exists(save_weight_path):
        model.load_state_dict(torch.load(save_weight_path, map_location=device))
    else:
        print("Warning: No checkpoint exceeded 0.05 Dice, using last epoch weights.")

    model.eval()
    dice_metric = DiceMetric(include_background=False, reduction="none")
    conf_metric = ConfusionMatrixMetric(include_background=False, metric_name=["precision", "recall"], reduction="none")
    hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="none")

    dices, precs, senss, hd95s = [], [], [], []
    with torch.no_grad():
        for test_batch in tqdm(test_loader, desc=f"Testing {model_name}"):
            t_in, t_lbl = test_batch["image"].to(device), test_batch["label"].to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                t_logits = sliding_window_inference(
                    t_in, roi_size=(96, 96, 96), sw_batch_size=2,
                    predictor=model_predictor, overlap=0.5, mode="gaussian"
                )
            t_pred = post_pred(torch.sigmoid(t_logits))

            gt_vol = torch.sum(t_lbl).item()
            pred_vol = torch.sum(t_pred).item()

            dice_metric(y_pred=t_pred, y=t_lbl)
            conf_metric(y_pred=t_pred, y=t_lbl)
            hd95_metric(y_pred=t_pred.cpu(), y=t_lbl.cpu())

            d_val = dice_metric.aggregate().item()
            if gt_vol == 0:
                d_val = 1.0 if pred_vol == 0 else 0.0
            elif np.isnan(d_val):
                d_val = 0.0

            c_res = conf_metric.aggregate()
            p_val = c_res[0].item() if not torch.isnan(c_res[0]) else (1.0 if (gt_vol == 0 and pred_vol == 0) else 0.0)
            s_val = c_res[1].item() if not torch.isnan(c_res[1]) else (1.0 if (gt_vol == 0 and pred_vol == 0) else 0.0)

            hd_val = hd95_metric.aggregate().item()
            hd95s.append(hd_val if not (np.isnan(hd_val) or np.isinf(hd_val)) else 373.0)

            dices.append(d_val)
            precs.append(p_val)
            senss.append(s_val)

            dice_metric.reset(); conf_metric.reset(); hd95_metric.reset()

    summary = {
        "Model": model_name,
        "Mean Dice": round(float(np.nanmean(dices)), 4),
        "Std Dice": round(float(np.nanstd(dices)), 4),
        "Precision": round(float(np.nanmean(precs)), 4),
        "Sensitivity": round(float(np.nanmean(senss)), 4),
        "HD95 (mm)": round(float(np.nanmean(hd95s)), 3),
    }
    print(f"Results for {model_name}: {summary}")
    return summary

## evaluation and results

In [8]:
baselines_to_run = [
    ("Standard_3D_UNet", get_standard_unet3d()),
    ("Attention_UNet_3D", get_attention_unet3d()),
    ("UNet_PlusPlus_3D", get_unet_plusplus3d()),
]

all_results = []

for name, model_inst in baselines_to_run:
    res = train_and_eval_baseline(
        model=model_inst,
        model_name=name,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=300
    )
    all_results.append(res)

# Display and save the comparison table
comparison_table = pd.DataFrame(all_results)
display(comparison_table)
comparison_table.to_csv("baselines_comparison_final.csv", index=False)

BasicUNetPlusPlus features: (16, 32, 64, 128, 256, 16).

🚀 Starting Training for Baseline: Standard_3D_UNet
  [Epoch 5] New Best Checkpoint Saved! Val Dice: 0.0036
  [Epoch 10] New Best Checkpoint Saved! Val Dice: 0.0040
Epoch [020/300] - Current Val Dice: 0.0038 (Best: 0.0040)
  [Epoch 25] New Best Checkpoint Saved! Val Dice: 0.0043
  [Epoch 30] New Best Checkpoint Saved! Val Dice: 0.0056
  [Epoch 35] New Best Checkpoint Saved! Val Dice: 0.0081
  [Epoch 40] New Best Checkpoint Saved! Val Dice: 0.0110
Epoch [040/300] - Current Val Dice: 0.0110 (Best: 0.0110)
  [Epoch 45] New Best Checkpoint Saved! Val Dice: 0.0124
  [Epoch 50] New Best Checkpoint Saved! Val Dice: 0.0140
  [Epoch 55] New Best Checkpoint Saved! Val Dice: 0.0163
  [Epoch 60] New Best Checkpoint Saved! Val Dice: 0.0172
Epoch [060/300] - Current Val Dice: 0.0172 (Best: 0.0172)
  [Epoch 65] New Best Checkpoint Saved! Val Dice: 0.0198
  [Epoch 70] New Best Checkpoint Saved! Val Dice: 0.0238
  [Epoch 75] New Best Checkpoint Sa

Testing Standard_3D_UNet:   0%|          | 0/38 [00:00<?, ?it/s]

Results for Standard_3D_UNet: {'Model': 'Standard_3D_UNet', 'Mean Dice': 0.2778, 'Std Dice': 0.2177, 'Precision': 0.1888, 'Sensitivity': 0.8368, 'HD95 (mm)': 44.843}

🚀 Starting Training for Baseline: Attention_UNet_3D
  [Epoch 5] New Best Checkpoint Saved! Val Dice: 0.0028
  [Epoch 10] New Best Checkpoint Saved! Val Dice: 0.0029
  [Epoch 15] New Best Checkpoint Saved! Val Dice: 0.0064
  [Epoch 20] New Best Checkpoint Saved! Val Dice: 0.0100
Epoch [020/300] - Current Val Dice: 0.0100 (Best: 0.0100)
  [Epoch 25] New Best Checkpoint Saved! Val Dice: 0.0133
  [Epoch 30] New Best Checkpoint Saved! Val Dice: 0.0202
  [Epoch 35] New Best Checkpoint Saved! Val Dice: 0.0251
  [Epoch 40] New Best Checkpoint Saved! Val Dice: 0.0285
Epoch [040/300] - Current Val Dice: 0.0285 (Best: 0.0285)
  [Epoch 45] New Best Checkpoint Saved! Val Dice: 0.0457
  [Epoch 50] New Best Checkpoint Saved! Val Dice: 0.0475
  [Epoch 55] New Best Checkpoint Saved! Val Dice: 0.0661
Epoch [060/300] - Current Val Dice: 0.0

Testing Attention_UNet_3D:   0%|          | 0/38 [00:00<?, ?it/s]

Results for Attention_UNet_3D: {'Model': 'Attention_UNet_3D', 'Mean Dice': 0.4343, 'Std Dice': 0.2673, 'Precision': 0.3393, 'Sensitivity': 0.8419, 'HD95 (mm)': 35.942}

🚀 Starting Training for Baseline: UNet_PlusPlus_3D
  [Epoch 5] New Best Checkpoint Saved! Val Dice: 0.0038
  [Epoch 10] New Best Checkpoint Saved! Val Dice: 0.0047
  [Epoch 15] New Best Checkpoint Saved! Val Dice: 0.0062
  [Epoch 20] New Best Checkpoint Saved! Val Dice: 0.0080
Epoch [020/300] - Current Val Dice: 0.0080 (Best: 0.0080)
  [Epoch 25] New Best Checkpoint Saved! Val Dice: 0.0122
  [Epoch 30] New Best Checkpoint Saved! Val Dice: 0.0194
  [Epoch 35] New Best Checkpoint Saved! Val Dice: 0.0298
  [Epoch 40] New Best Checkpoint Saved! Val Dice: 0.0406
Epoch [040/300] - Current Val Dice: 0.0406 (Best: 0.0406)
  [Epoch 45] New Best Checkpoint Saved! Val Dice: 0.0507
  [Epoch 50] New Best Checkpoint Saved! Val Dice: 0.0678
  [Epoch 55] New Best Checkpoint Saved! Val Dice: 0.0813
  [Epoch 60] New Best Checkpoint Saved

Testing UNet_PlusPlus_3D:   0%|          | 0/38 [00:00<?, ?it/s]

Results for UNet_PlusPlus_3D: {'Model': 'UNet_PlusPlus_3D', 'Mean Dice': 0.5148, 'Std Dice': 0.253, 'Precision': 0.4199, 'Sensitivity': 0.8455, 'HD95 (mm)': 37.004}


,Model,Mean Dice,Std Dice,Precision,Sensitivity,HD95 (mm)
0,Standard_3D_UNet,0.2778,0.2177,0.1888,0.8368,44.843
1,Attention_UNet_3D,0.4343,0.2673,0.3393,0.8419,35.942
2,UNet_PlusPlus_3D,0.5148,0.2530,0.4199,0.8455,37.004
